In [1]:
# ============================================================
# Stage 06 - Error Analysis & Implicit Descriptions
# Step 6.1 - Setup & Load Predictions
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Locate project directory
CURRENT_DIR = Path.cwd().resolve()

PROJECT_DIR = next(
    (
        p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (p / "data" / "processed" / "test.csv").exists()
    ),
    None
)

assert PROJECT_DIR is not None, "Project directory not found!"

RESULTS_DIR = PROJECT_DIR / "results"
PREDICTIONS_PATH = RESULTS_DIR / "test_predictions.csv"

assert PREDICTIONS_PATH.exists(), (
    "test_predictions.csv not found! "
    "Run Step 5.4 in 05_Evaluation.ipynb first."
)

# Load saved predictions
df = pd.read_csv(
    PREDICTIONS_PATH,
    dtype=str,
    keep_default_na=False
)

# Required columns
required_columns = {
    "CAMPNO",
    "text",
    "actual_label",
    "traditional_prediction",
    "minilm_prediction"
}

assert required_columns.issubset(df.columns)

# Recalculate correctness
df["traditional_correct"] = (
    df["actual_label"] == df["traditional_prediction"]
)

df["minilm_correct"] = (
    df["actual_label"] == df["minilm_prediction"]
)

# Validate results
assert len(df) == 763
assert df["traditional_correct"].sum() == 739
assert df["minilm_correct"].sum() == 719
assert df["CAMPNO"].is_unique

print("=" * 55)
print("STAGE 06 - ERROR ANALYSIS SETUP")
print("=" * 55)

print("Total Test Examples:", len(df))
print("Number of Classes:", df["actual_label"].nunique())

print("\nTraditional Correct:",
      df["traditional_correct"].sum())

print("Traditional Errors:",
      (~df["traditional_correct"]).sum())

print("\nMiniLM Correct:",
      df["minilm_correct"].sum())

print("MiniLM Errors:",
      (~df["minilm_correct"]).sum())

print("\nData Loading Verification: PASSED!")


STAGE 06 - ERROR ANALYSIS SETUP
Total Test Examples: 763
Number of Classes: 5

Traditional Correct: 739
Traditional Errors: 24

MiniLM Correct: 719
MiniLM Errors: 44

Data Loading Verification: PASSED!


In [2]:

# ============================================================
# Step 6.1 - Compare Model Prediction Errors
# ============================================================

# Both models correct
both_correct = (
    df["traditional_correct"]
    & df["minilm_correct"]
)

# Both models incorrect
both_wrong = (
    ~df["traditional_correct"]
    & ~df["minilm_correct"]
)

# Traditional correct, MiniLM wrong
traditional_only_correct = (
    df["traditional_correct"]
    & ~df["minilm_correct"]
)

# MiniLM correct, Traditional wrong
minilm_only_correct = (
    ~df["traditional_correct"]
    & df["minilm_correct"]
)

# Create comparison summary
error_summary = pd.DataFrame({
    "Category": [
        "Both Correct",
        "Both Wrong",
        "Traditional Only Correct",
        "MiniLM Only Correct"
    ],
    "Count": [
        both_correct.sum(),
        both_wrong.sum(),
        traditional_only_correct.sum(),
        minilm_only_correct.sum()
    ]
})

print("=" * 55)
print("MODEL ERROR COMPARISON")
print("=" * 55)

display(error_summary)

# Verify all examples are accounted for
assert error_summary["Count"].sum() == len(df)

print("\nDifferent Predictions:", (
    df["traditional_prediction"]
    != df["minilm_prediction"]
).sum())

# Show real examples where both models failed
print("\n" + "=" * 55)
print("EXAMPLES WHERE BOTH MODELS FAILED")
print("=" * 55)

error_examples = df.loc[
    both_wrong,
    [
        "CAMPNO",
        "text",
        "actual_label",
        "traditional_prediction",
        "minilm_prediction"
    ]
]

for _, row in error_examples.head(5).iterrows():

    print("\n" + "-" * 55)
    print("Campaign:", row["CAMPNO"])
    print("Actual:", row["actual_label"])
    print("Traditional:", row["traditional_prediction"])
    print("MiniLM:", row["minilm_prediction"])
    print("Description:", row["text"][:500])

print("\nError Comparison: COMPLETED!")


MODEL ERROR COMPARISON


,Category,Count
0,Both Correct,708
1,Both Wrong,13
2,Traditional Only Correct,31
3,MiniLM Only Correct,11



Different Predictions: 43

EXAMPLES WHERE BOTH MODELS FAILED

-------------------------------------------------------
Campaign: 26V253000
Actual: Electrical System
Traditional: Service Brakes
MiniLM: Service Brakes
Description: Hexagon Purus Systems USA LLC (Hexagon Purus) is recalling certain 2025-2026 Tern RC8 vehicles. A software error may occur between the digital display cluster and the pressure sensors, causing the system to show the incorrect air brake pressure and falsely indicate a brake system failure. As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety Standard (FMVSS) number 121, "Air Brake Systems."

-------------------------------------------------------
Campaign: 17V689000
Actual: Service Brakes
Traditional: Electrical System
MiniLM: Electrical System
Description: Heartland Recreational Vehicles, LLC (Heartland) is recalling certain 2018 Prowler and Trail Runner recreational trailers. The wires for the electric brakes may get cau

In [3]:

# ============================================================
# Step 6.2 - Define Implicit Description Rules
# ============================================================

import re
import numpy as np
import pandas as pd

# Fixed lexical patterns for each target class
COMPONENT_PATTERNS = {
    "Electrical System": r"\belectric(?:al)?\b",
    "Air Bags": r"\bair[\s-]*bags?\b",
    "Service Brakes": r"\bbrak(?:e|es|ing|ed)\b",
    "Steering": r"\bsteer(?:ing|s|ed)?\b",
    "Fuel System": r"\bfuels?\b"
}

# Compile patterns (case-insensitive)
compiled_patterns = {
    label: re.compile(pattern, re.IGNORECASE)
    for label, pattern in COMPONENT_PATTERNS.items()
}

# Check whether the description directly names its true class
def has_explicit_component(text, actual_label):

    pattern = compiled_patterns[actual_label]

    return bool(pattern.search(text))


# Detect explicit versus implicit descriptions
df["explicit_component"] = df.apply(
    lambda row: has_explicit_component(
        row["text"],
        row["actual_label"]
    ),
    axis=1
)

df["description_type"] = np.where(
    df["explicit_component"],
    "Explicit",
    "Implicit"
)

# Verification
assert df["description_type"].isin(
    ["Explicit", "Implicit"]
).all()

assert len(df) == 763

print("=" * 55)
print("IMPLICIT DESCRIPTION DETECTION")
print("=" * 55)

print("\nDescription Distribution:")
print(df["description_type"].value_counts())

print("\nDescription Percentages:")
print(
    df["description_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nImplicit Detection: COMPLETED!")


IMPLICIT DESCRIPTION DETECTION

Description Distribution:
description_type
Explicit    519
Implicit    244
Name: count, dtype: int64

Description Percentages:
description_type
Explicit    68.02
Implicit    31.98
Name: proportion, dtype: float64

Implicit Detection: COMPLETED!


In [4]:

# ============================================================
# Step 6.2B - Inspect Implicit Descriptions
# ============================================================

print("=" * 55)
print("IMPLICIT DATASET ANALYSIS")
print("=" * 55)

# 1. Class distribution
distribution = pd.crosstab(
    df["actual_label"],
    df["description_type"]
)

distribution["Total"] = distribution.sum(axis=1)

distribution["Implicit %"] = (
    distribution["Implicit"]
    / distribution["Total"]
    * 100
).round(2)

print("\nImplicit vs Explicit by Class:")
display(distribution)

# 2. Select implicit examples
implicit_df = df[
    df["description_type"] == "Implicit"
].copy()

print("\nTotal Implicit Examples:", len(implicit_df))

# 3. Show two examples from each class
print("\n" + "=" * 55)
print("IMPLICIT DESCRIPTION EXAMPLES")
print("=" * 55)

for label in COMPONENT_PATTERNS:

    examples = implicit_df[
        implicit_df["actual_label"] == label
    ].head(2)

    print(f"\nCLASS: {label}")

    for _, row in examples.iterrows():

        print("-" * 55)
        print("Campaign:", row["CAMPNO"])
        print("Text:", row["text"][:350])

print("\nImplicit Dataset Inspection: COMPLETED!")


IMPLICIT DATASET ANALYSIS

Implicit vs Explicit by Class:


description_type,Explicit,Implicit,Total,Implicit %
actual_label,,,,
Air Bags,109,9,118,7.63
Electrical System,74,210,284,73.94
Fuel System,105,6,111,5.41
Service Brakes,138,9,147,6.12
Steering,93,10,103,9.71



Total Implicit Examples: 244

IMPLICIT DESCRIPTION EXAMPLES

CLASS: Electrical System
-------------------------------------------------------
Campaign: 23V204000
Text: Daimler Vans USA, LLC (DVUSA) is recalling certain 2019-2022 Mercedes-Benz Sprinter and 2019-2021 Freightliner Sprinter vehicles. The instrument cluster may fail to display any information when the vehicle is started.
-------------------------------------------------------
Campaign: 26V144000
Text: Gillig, LLC (Gillig) is recalling certain 2023-2026 Low Floor transit buses. The high voltage cable, connecting the inverter to the drive unit, contains a splice connection that reduces the amp capacity rating.

CLASS: Air Bags
-------------------------------------------------------
Campaign: 13V607000
Text: Jaguar Land Rover North America, LLC (Land Rover) is recalling certain model year 2013-2014 Range Rover vehicles. In the affected vehicles, due to insufficient clearance, the Supplemental Restraint System (SRS) connector 

In [5]:

# ============================================================
# Step 6.3 - Implicit vs Explicit Performance
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score

CLASS_LABELS = sorted(df["actual_label"].unique())

# Verify group distribution
print("=" * 60)
print("CLASS DISTRIBUTION BY DESCRIPTION TYPE")
print("=" * 60)

class_distribution = pd.crosstab(
    df["actual_label"],
    df["description_type"]
).reindex(
    index=CLASS_LABELS,
    columns=["Explicit", "Implicit"],
    fill_value=0
)

display(class_distribution)

# Calculate metrics for both groups
results = []

for description_type in ["Explicit", "Implicit"]:

    subset = df[
        df["description_type"] == description_type
    ].copy()

    y_true = subset["actual_label"]

    for model_name, prediction_col in [
        ("TF-IDF + LR", "traditional_prediction"),
        ("MiniLM + LR", "minilm_prediction")
    ]:

        y_pred = subset[prediction_col]

        accuracy = accuracy_score(y_true, y_pred)

        # Compute macro F1 only when all classes are present
        if set(CLASS_LABELS).issubset(set(y_true)):
            macro_f1 = f1_score(
                y_true,
                y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0
            )
        else:
            macro_f1 = np.nan

        correct = int((y_true == y_pred).sum())

        results.append({
            "Description Type": description_type,
            "Model": model_name,
            "Examples": len(subset),
            "Correct": correct,
            "Errors": len(subset) - correct,
            "Accuracy": accuracy,
            "Macro F1": macro_f1
        })

# Results table
performance_df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("EXPLICIT VS IMPLICIT PERFORMANCE")
print("=" * 60)

display(performance_df.round(4))

# ============================================================
# Compare models on implicit descriptions
# ============================================================

implicit_results = performance_df[
    performance_df["Description Type"] == "Implicit"
].set_index("Model")

traditional_acc = implicit_results.loc[
    "TF-IDF + LR", "Accuracy"
]

minilm_acc = implicit_results.loc[
    "MiniLM + LR", "Accuracy"
]

accuracy_gap = (
    traditional_acc - minilm_acc
) * 100

print("\n" + "=" * 60)
print("IMPLICIT DESCRIPTION COMPARISON")
print("=" * 60)

print(f"TF-IDF Accuracy: {traditional_acc:.4f}")
print(f"MiniLM Accuracy: {minilm_acc:.4f}")

print(
    f"Accuracy Difference: {accuracy_gap:+.2f} "
    "percentage points (TF-IDF minus MiniLM)"
)

if not performance_df["Macro F1"].isna().any():

    traditional_f1 = implicit_results.loc[
        "TF-IDF + LR", "Macro F1"
    ]

    minilm_f1 = implicit_results.loc[
        "MiniLM + LR", "Macro F1"
    ]

    print(f"\nTF-IDF Macro F1: {traditional_f1:.4f}")
    print(f"MiniLM Macro F1: {minilm_f1:.4f}")

    print(
        "Macro F1 Difference:",
        round(traditional_f1 - minilm_f1, 4)
    )
else:
    print("\nMacro F1 requires review: a subset is missing classes.")

# Verification
assert len(df) == 763
assert class_distribution.to_numpy().sum() == 763
assert performance_df["Examples"].isin([519, 244]).all()

print("\nImplicit Performance Analysis: COMPLETED!")


CLASS DISTRIBUTION BY DESCRIPTION TYPE


description_type,Explicit,Implicit
actual_label,,
Air Bags,109,9
Electrical System,74,210
Fuel System,105,6
Service Brakes,138,9
Steering,93,10



EXPLICIT VS IMPLICIT PERFORMANCE


,Description Type,Model,Examples,Correct,Errors,Accuracy,Macro F1
0,Explicit,TF-IDF + LR,519,511,8,0.9846,0.9831
1,Explicit,MiniLM + LR,519,504,15,0.9711,0.9674
2,Implicit,TF-IDF + LR,244,228,16,0.9344,0.7982
3,Implicit,MiniLM + LR,244,215,29,0.8811,0.6756



IMPLICIT DESCRIPTION COMPARISON
TF-IDF Accuracy: 0.9344
MiniLM Accuracy: 0.8811
Accuracy Difference: +5.33 percentage points (TF-IDF minus MiniLM)

TF-IDF Macro F1: 0.7982
MiniLM Macro F1: 0.6756
Macro F1 Difference: 0.1226

Implicit Performance Analysis: COMPLETED!


In [6]:

# ============================================================
# Step 6.4A - Implicit Error Breakdown by Class
# ============================================================

import pandas as pd

implicit_df = df[
    df["description_type"] == "Implicit"
].copy()

results_by_class = []

for label in sorted(implicit_df["actual_label"].unique()):

    subset = implicit_df[
        implicit_df["actual_label"] == label
    ]

    total = len(subset)

    traditional_correct = int(
        subset["traditional_correct"].sum()
    )

    minilm_correct = int(
        subset["minilm_correct"].sum()
    )

    results_by_class.append({
        "Class": label,
        "Examples": total,
        "Traditional Correct": traditional_correct,
        "Traditional Errors": total - traditional_correct,
        "Traditional Recall": traditional_correct / total,
        "MiniLM Correct": minilm_correct,
        "MiniLM Errors": total - minilm_correct,
        "MiniLM Recall": minilm_correct / total
    })

class_results = pd.DataFrame(results_by_class)

print("=" * 60)
print("IMPLICIT PERFORMANCE BY CLASS")
print("=" * 60)

display(class_results.round(4))

# Verify results
assert class_results["Examples"].sum() == 244
assert class_results["Traditional Correct"].sum() == 228
assert class_results["MiniLM Correct"].sum() == 215

print("\n" + "=" * 60)
print("IMPLICIT ERROR TOTALS")
print("=" * 60)

print(
    "Traditional Errors:",
    class_results["Traditional Errors"].sum()
)

print(
    "MiniLM Errors:",
    class_results["MiniLM Errors"].sum()
)

print("\nImplicit Class Analysis: PASSED!")


IMPLICIT PERFORMANCE BY CLASS


,Class,Examples,Traditional Correct,Traditional Errors,Traditional Recall,MiniLM Correct,MiniLM Errors,MiniLM Recall
0,Air Bags,9,9,0,1.0000,7,2,0.7778
1,Electrical System,210,201,9,0.9571,191,19,0.9095
2,Fuel System,6,5,1,0.8333,5,1,0.8333
3,Service Brakes,9,6,3,0.6667,4,5,0.4444
4,Steering,10,7,3,0.7000,8,2,0.8000



IMPLICIT ERROR TOTALS
Traditional Errors: 16
MiniLM Errors: 29

Implicit Class Analysis: PASSED!


In [7]:

# ============================================================
# Step 6.4B - Select Real Error Cases
# ============================================================

import numpy as np
import pandas as pd

# 1. Select implicit descriptions with at least one error
error_candidates = df[
    (df["description_type"] == "Implicit")
    & (
        (~df["traditional_correct"])
        | (~df["minilm_correct"])
    )
].copy()

# 2. Classify error types
trad_wrong = ~error_candidates["traditional_correct"]
mini_wrong = ~error_candidates["minilm_correct"]

error_candidates["error_type"] = np.select(
    [
        trad_wrong & mini_wrong,
        trad_wrong & ~mini_wrong,
        ~trad_wrong & mini_wrong
    ],
    [
        "Both Models Wrong",
        "Traditional Wrong - MiniLM Correct",
        "MiniLM Wrong - Traditional Correct"
    ],
    default="Other"
)

print("=" * 60)
print("IMPLICIT ERROR CATEGORIES")
print("=" * 60)

print(error_candidates["error_type"].value_counts())

# 3. Prioritize different component classes
class_priority = {
    "Service Brakes": 0,
    "Steering": 1,
    "Fuel System": 2,
    "Air Bags": 3,
    "Electrical System": 4
}

error_candidates["priority"] = (
    error_candidates["actual_label"]
    .map(class_priority)
    .fillna(99)
)

error_candidates = error_candidates.sort_values(
    ["priority", "CAMPNO"]
)

# 4. Select one example from each error type if available
selected_indices = []

error_types = [
    "Both Models Wrong",
    "Traditional Wrong - MiniLM Correct",
    "MiniLM Wrong - Traditional Correct"
]

for error_type in error_types:
    subset = error_candidates[
        error_candidates["error_type"] == error_type
    ]

    if not subset.empty:
        selected_indices.append(subset.index[0])

# 5. Fill any remaining slots, preferring different classes
for idx, row in error_candidates.iterrows():

    if len(selected_indices) >= 3:
        break

    selected_classes = set(
        error_candidates.loc[
            selected_indices, "actual_label"
        ]
    )

    if (
        idx not in selected_indices
        and row["actual_label"] not in selected_classes
    ):
        selected_indices.append(idx)

for idx in error_candidates.index:
    if len(selected_indices) >= 3:
        break
    if idx not in selected_indices:
        selected_indices.append(idx)

selected_errors = error_candidates.loc[
    selected_indices
].copy()

# 6. Print complete descriptions for manual analysis
print("\n" + "=" * 60)
print("SELECTED ERROR CASES")
print("=" * 60)

for number, (_, row) in enumerate(
    selected_errors.iterrows(), start=1
):
    print("\n" + "=" * 60)
    print(f"CASE {number}")
    print("=" * 60)

    print("Campaign:", row["CAMPNO"])
    print("Error Type:", row["error_type"])
    print("Actual Label:", row["actual_label"])
    print("Traditional Prediction:",
          row["traditional_prediction"])
    print("MiniLM Prediction:",
          row["minilm_prediction"])

    print("\nFULL DESCRIPTION:")
    print(row["text"])

print("\nError Case Selection: COMPLETED!")


IMPLICIT ERROR CATEGORIES
error_type
MiniLM Wrong - Traditional Correct    18
Both Models Wrong                     11
Traditional Wrong - MiniLM Correct     5
Name: count, dtype: int64

SELECTED ERROR CASES

CASE 1
Campaign: 10V249000
Error Type: Both Models Wrong
Actual Label: Service Brakes
Traditional Prediction: Electrical System
MiniLM Prediction: Steering

FULL DESCRIPTION:
DAIMLER TRUCKS IS RECALLING CERTAIN MODEL YEAR 2008-2009 STERLING 360 VEHICLES MANUFACTURED FROM AUGUST 8, 2007 THROUGH JULY 31, 2008. THE GROUND HARNESS CONNECTOR FOR THE ABS HYDRAULIC UNIT MAY BECOME CORRODED DUE INTRUSION POTENTIALLY CAUSING THE ABS HYDRAULIC MOTOR TO RUN CONTINUOUSLY.

CASE 2
Campaign: 13V055000
Error Type: Traditional Wrong - MiniLM Correct
Actual Label: Steering
Traditional Prediction: Electrical System
MiniLM Prediction: Steering

FULL DESCRIPTION:
Prevost is recalling certain model year 1993 thru 2013 model H3-40, H3-41, H3-45, and H3-45 VIP Prevost coaches manufactured from December 

In [8]:

# ============================================================
# Step 6.5 - Save Final Error Analysis
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Collect all test examples with at least one error
all_errors = df.loc[
    (~df["traditional_correct"])
    | (~df["minilm_correct"])
].copy()

# 2. Classify error types
trad_wrong = ~all_errors["traditional_correct"]
mini_wrong = ~all_errors["minilm_correct"]

all_errors["error_type"] = np.select(
    [
        trad_wrong & mini_wrong,
        trad_wrong & ~mini_wrong,
        ~trad_wrong & mini_wrong
    ],
    [
        "Both Models Wrong",
        "Traditional Wrong - MiniLM Correct",
        "MiniLM Wrong - Traditional Correct"
    ],
    default="Other"
)

# 3. Human-reviewed explanations for selected cases
explanations = [
    (
        "The description mentions both an electrical "
        "connector and an ABS hydraulic unit. These "
        "overlapping system details may contribute "
        "to classification confusion."
    ),
    (
        "The description mentions drag link and pitman "
        "arm components without explicitly using the "
        "word steering. MiniLM classified this case "
        "correctly, while TF-IDF did not."
    ),
    (
        "The description mentions a slack adjuster "
        "housing, a component associated with the "
        "braking system. TF-IDF classified the case "
        "correctly, while MiniLM predicted Steering."
    )
]

assert len(selected_errors) == 3

# 4. Add explanations to selected examples
selected_review = selected_errors.copy()
selected_review["error_reason"] = explanations

# 5. Match explanations to all errors by campaign
reason_map = dict(
    zip(
        selected_review["CAMPNO"],
        selected_review["error_reason"]
    )
)

all_errors["error_reason"] = (
    all_errors["CAMPNO"]
    .map(reason_map)
    .fillna("")
)

# 6. Select final columns
columns = [
    "CAMPNO",
    "text",
    "actual_label",
    "traditional_prediction",
    "minilm_prediction",
    "description_type",
    "error_type",
    "error_reason"
]

# 7. Save both CSV files
all_errors[columns].to_csv(
    RESULTS_DIR / "error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

selected_review[columns].to_csv(
    RESULTS_DIR / "selected_error_cases.csv",
    index=False,
    encoding="utf-8-sig"
)

# 8. Verification
assert len(all_errors) == 55
assert len(selected_review) == 3

assert (
    all_errors[
        all_errors["description_type"] == "Implicit"
    ].shape[0] == 34
)

print("=" * 55)
print("FINAL ERROR ANALYSIS")
print("=" * 55)

print("Total Examples With Errors:", len(all_errors))
print("Implicit Examples With Errors:", 34)
print("Selected Reviewed Cases:", len(selected_review))

print("\nSaved Files:")
print("- error_analysis.csv")
print("- selected_error_cases.csv")

print("\nError Analysis Saving: PASSED!")


FINAL ERROR ANALYSIS
Total Examples With Errors: 55
Implicit Examples With Errors: 34
Selected Reviewed Cases: 3

Saved Files:
- error_analysis.csv
- selected_error_cases.csv

Error Analysis Saving: PASSED!
